# F26-07 | Phase 4: Live Training & Evaluation of Improved Biomedical Transformers
## Domain Pre-Training (PubMedBERT), Structural Position Conditioning, Class-Weighted Loss, and Algorithmic Fairness on PubMed 20k RCT

**Group F26-07:** Azan Wasty (24L-2557), Nauman Iqbal (24L-2565), Afras Shahnawaz (24L-2547)  
**Instructor:** Hajra Waheed | **Department of Data Science, FAST-NUCES**  

---

### 🔬 100% Live Execution Guide:
This notebook executes the entire training and evaluation pipeline from scratch on your active GPU session:
1. Ingests raw PubMed 20k RCT text files (`train.txt`, `dev.txt`, `test.txt`).
2. Injects position-aware structural prompts (`[LINE i OF n] [PCT p%] <text>`).
3. Computes inverse class weights for balanced cross-entropy loss.
4. Fine-tunes `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract` using HuggingFace Trainer.
5. Executes live test inference across all 30,135 sentences.
6. Directly computes Accuracy, Macro F1, Weighted F1, Cohen's Kappa, and per-class metrics.
7. Runs live error analysis across abstract position deciles ($d=0\dots9$) and length quartiles.
8. Computes algorithmic fairness audit and statistical significance tests.
9. Exports all verified results directly to CSV files.

## 1. Environment Setup & GPU Verification

In [ ]:
# Install dependencies
%pip install -q transformers datasets accelerate xgboost scikit-learn seaborn matplotlib

In [ ]:
import os, re, time, math, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from pathlib import Path
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    classification_report, confusion_matrix, precision_recall_fscore_support
)
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
sns.set_theme(style="whitegrid")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Active Compute Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

# Set output directory (Google Drive or local)
OUTPUT_DIR = Path("./results_phase4_live")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[*] Outputs will be written to: {OUTPUT_DIR.resolve()}")

## 2. Ingestion & Preprocessing of PubMed 20k RCT

In [ ]:
# If running in Colab and raw files are not present, clone the official PubMed RCT repo
raw_dir = Path('data/raw')
if not (raw_dir / 'train.txt').exists():
    !git clone https://github.com/Franck-Dernoncourt/pubmed-rct.git
    raw_dir = Path('pubmed-rct/PubMed_20k_RCT')

LABELS = ['BACKGROUND', 'OBJECTIVE', 'METHODS', 'RESULTS', 'CONCLUSIONS']
LABEL2ID = {lbl: i for i, lbl in enumerate(LABELS)}
ID2LABEL = {i: lbl for i, lbl in enumerate(LABELS)}

def clean_text(text: str) -> str:
    return re.sub(r'\s+', ' ', str(text)).strip()

def parse_pubmed_rct(filepath: Path) -> pd.DataFrame:
    rows, current_abstract, abstract_id = [], [], None
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line.startswith('###'):
                if current_abstract and abstract_id:
                    tot = len(current_abstract)
                    for i, (lbl, txt) in enumerate(current_abstract):
                        c_txt = clean_text(txt)
                        rows.append({
                            'abstract_id': abstract_id, 'label': lbl, 'text': c_txt,
                            'line_number': i, 'total_lines': tot, 'line_length': len(c_txt),
                            'relative_position': i / tot if tot > 0 else 0.0
                        })
                abstract_id = line[3:].strip()
                current_abstract = []
            elif line == '':
                if current_abstract and abstract_id:
                    tot = len(current_abstract)
                    for i, (lbl, txt) in enumerate(current_abstract):
                        c_txt = clean_text(txt)
                        rows.append({
                            'abstract_id': abstract_id, 'label': lbl, 'text': c_txt,
                            'line_number': i, 'total_lines': tot, 'line_length': len(c_txt),
                            'relative_position': i / tot if tot > 0 else 0.0
                        })
                current_abstract = []
            else:
                parts = line.split('\t', 1)
                if len(parts) == 2:
                    current_abstract.append((parts[0].strip(), parts[1].strip()))
    return pd.DataFrame(rows)

train_df = parse_pubmed_rct(raw_dir / 'train.txt')
dev_df = parse_pubmed_rct(raw_dir / 'dev.txt')
test_df = parse_pubmed_rct(raw_dir / 'test.txt')

print(f"train: {len(train_df):,} sentences | {train_df['abstract_id'].nunique():,} abstracts")
print(f"dev  : {len(dev_df):,} sentences | {dev_df['abstract_id'].nunique():,} abstracts")
print(f"test : {len(test_df):,} sentences | {test_df['abstract_id'].nunique():,} abstracts")

## 3. Position-Conditioned Prompt Building & Class Weight Calculation

In [ ]:
# Format position prompts
def format_position_prompt(row: pd.Series) -> str:
    line_num = int(row['line_number'])
    total = int(row['total_lines'])
    pct = int(round(float(row['relative_position']) * 100))
    return f"[LINE {line_num} OF {total}] [PCT {pct}%] {row['text']}"

train_texts = [format_position_prompt(r) for _, r in train_df.iterrows()]
dev_texts = [format_position_prompt(r) for _, r in dev_df.iterrows()]
test_texts = [format_position_prompt(r) for _, r in test_df.iterrows()]

y_train = [LABEL2ID[l] for l in train_df['label']]
y_dev = [LABEL2ID[l] for l in dev_df['label']]
y_test = np.array([LABEL2ID[l] for l in test_df['label']])

# Compute inverse class weights
weights = compute_class_weight('balanced', classes=np.arange(len(LABELS)), y=np.array(y_train))
class_weights_t = torch.tensor(weights, dtype=torch.float)
print("Computed Balanced Class Weights:")
for lbl, w in zip(LABELS, weights):
    print(f"  {lbl:12s}: {w:.4f}")

## 4. Model Architecture & Weighted Trainer Setup

In [ ]:
MODEL_NAME = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract'
print(f"[*] Loading tokenizer and model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PositionDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.labels = labels
        self.encodings = tokenizer(texts, truncation=True, max_length=max_length, padding=False)
    def __len__(self):
        return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['label'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Set optional quick-test subsample switch (set to None for complete 180k run)
TRAIN_SUBSAMPLE = None  # e.g., set to 20000 for a rapid 10-minute check
if TRAIN_SUBSAMPLE is not None:
    sub_idx = np.random.choice(len(train_texts), size=TRAIN_SUBSAMPLE, replace=False)
    train_texts_sub = [train_texts[i] for i in sub_idx]
    y_train_sub = [y_train[i] for i in sub_idx]
    train_ds = PositionDataset(train_texts_sub, y_train_sub, tokenizer)
else:
    train_ds = PositionDataset(train_texts, y_train, tokenizer)

dev_ds = PositionDataset(dev_texts, y_dev, tokenizer)
test_ds = PositionDataset(test_texts, None, tokenizer)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fn = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device)) if self.class_weights is not None else nn.CrossEntropyLoss()
        loss = loss_fn(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID
)

## 5. Live Model Training

In [ ]:
EPOCHS = 5
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LR = 2e-5

steps_per_epoch = math.ceil(len(train_ds) / BATCH_SIZE)
warmup_steps = int(0.06 * steps_per_epoch * EPOCHS)

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'ckpt'),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    fp16=torch.cuda.is_available(),
    seed=SEED,
    logging_steps=100,
    report_to='none',
)

def compute_metrics_eval(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'weighted_f1': float(f1_score(labels, preds, average='weighted', zero_division=0)),
        'macro_f1': float(f1_score(labels, preds, average='macro', zero_division=0)),
        'cohen_kappa': float(cohen_kappa_score(labels, preds))
    }

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=collator,
    compute_metrics=compute_metrics_eval,
    class_weights=class_weights_t,
)

t0 = time.time()
trainer.train()
train_time_sec = time.time() - t0
print(f"[*] Training Finished in {train_time_sec:.1f}s ({train_time_sec/60:.1f} mins)")

## 6. Live Inference & Overall Test Metrics

In [ ]:
t_inf = time.time()
pred_output = trainer.predict(test_ds)
infer_time_sec = time.time() - t_inf
infer_per_1k = (infer_time_sec / len(test_ds)) * 1000.0

logits = pred_output.predictions
y_pred = np.argmax(logits, axis=-1)

# Save prediction arrays
np.save(str(OUTPUT_DIR / 'test_predictions.npy'), y_pred)
np.save(str(OUTPUT_DIR / 'test_logits.npy'), logits)

# Compute live metrics
acc = float(accuracy_score(y_test, y_pred))
w_f1 = float(f1_score(y_test, y_pred, average='weighted', zero_division=0))
m_f1 = float(f1_score(y_test, y_pred, average='macro', zero_division=0))
kappa = float(cohen_kappa_score(y_test, y_pred))

live_metrics_df = pd.DataFrame([{
    'Model': 'PubMedBERT + Position + Weighted Loss',
    'Test_Accuracy': round(acc, 4),
    'Weighted_F1': round(w_f1, 4),
    'Macro_F1': round(m_f1, 4),
    'Cohen_Kappa': round(kappa, 4),
    'Train_Time_s': round(train_time_sec, 1),
    'Inference_s_per_1k': round(infer_per_1k, 3),
    'Hardware': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
}])
live_metrics_df.to_csv(OUTPUT_DIR / 'live_overall_metrics.csv', index=False)
display(live_metrics_df)

## 7. Live Per-Class Performance Breakdown

In [ ]:
p, r, f, s = precision_recall_fscore_support(y_test, y_pred, labels=range(len(LABELS)), zero_division=0)
live_per_class_df = pd.DataFrame({
    'Rhetorical_Role': LABELS,
    'Precision': np.round(p, 4),
    'Recall': np.round(r, 4),
    'F1': np.round(f, 4),
    'Support': s
})
live_per_class_df.to_csv(OUTPUT_DIR / 'live_per_class_metrics.csv', index=False)
display(live_per_class_df)

## 8. Live Diagnostic Error Analysis (Position Deciles & Length Quartiles)

In [ ]:
# 1. Positional Deciles (d=0 to 9)
rel_pos = test_df['relative_position'].values
pos_bins = np.minimum((rel_pos * 10).astype(int), 9)
pos_rows = []
for b in range(10):
    m = (pos_bins == b)
    tot = int(m.sum())
    corr = int((y_test[m] == y_pred[m]).sum())
    pos_rows.append({
        'Position_Decile': b,
        'Range': f"{b*10}-{(b+1)*10}%",
        'Total_Sentences': tot,
        'Accuracy': round(corr / tot, 4) if tot > 0 else 0.0,
        'Error_Rate': round((tot - corr) / tot, 4) if tot > 0 else 0.0
    })
live_pos_df = pd.DataFrame(pos_rows)
live_pos_df.to_csv(OUTPUT_DIR / 'live_error_analysis_position.csv', index=False)

# 2. Sentence Length Quartiles
lengths = test_df['line_length'].values
q_labels = ['Q1 (Shortest)', 'Q2 (Med-Short)', 'Q3 (Med-Long)', 'Q4 (Longest)']
quartiles = pd.qcut(lengths, q=4, labels=q_labels, duplicates='drop')
len_rows = []
for ql in q_labels:
    m = (quartiles == ql)
    tot = int(m.sum())
    corr = int((y_test[m] == y_pred[m]).sum())
    len_rows.append({
        'Length_Quartile': ql,
        'Total_Sentences': tot,
        'Accuracy': round(corr / tot, 4) if tot > 0 else 0.0,
        'Error_Rate': round((tot - corr) / tot, 4) if tot > 0 else 0.0
    })
live_len_df = pd.DataFrame(len_rows)
live_len_df.to_csv(OUTPUT_DIR / 'live_error_analysis_length.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.lineplot(data=live_pos_df, x='Position_Decile', y='Accuracy', marker='o', color='purple', ax=axes[0])
axes[0].set_title('Live Accuracy across Position Deciles (0-9)')
sns.barplot(data=live_len_df, x='Length_Quartile', y='Accuracy', palette='Blues_d', ax=axes[1])
axes[1].set_title('Live Accuracy across Sentence Length Quartiles')
plt.tight_layout()
plt.show()

## 9. Live Algorithmic Bias & Fairness Audit

In [ ]:
# Recall Parity (Minority OBJECTIVE vs Majority METHODS)
rec_obj = float(live_per_class_df.loc[live_per_class_df['Rhetorical_Role'] == 'OBJECTIVE', 'Recall'].values[0])
rec_meth = float(live_per_class_df.loc[live_per_class_df['Rhetorical_Role'] == 'METHODS', 'Recall'].values[0])
recall_disparity = round(rec_obj / rec_meth, 4) if rec_meth > 0 else 0.0

# Boundary vs Interior Parity
b_mask = (rel_pos < 0.10) | (rel_pos >= 0.90)
acc_b = float((y_test[b_mask] == y_pred[b_mask]).mean())
acc_i = float((y_test[~b_mask] == y_pred[~b_mask]).mean())
boundary_disparity = round(acc_b / acc_i, 4) if acc_i > 0 else 0.0

# Length Parity
med_l = float(np.median(lengths))
s_mask = (lengths <= med_l)
acc_short = float((y_test[s_mask] == y_pred[s_mask]).mean())
acc_long = float((y_test[~s_mask] == y_pred[~s_mask]).mean())
length_disparity = round(min(acc_short, acc_long) / max(acc_short, acc_long), 4)

live_fairness_df = pd.DataFrame([{
    'Minority_Role': 'OBJECTIVE',
    'Minority_Recall': rec_obj,
    'Majority_Role': 'METHODS',
    'Majority_Recall': rec_meth,
    'Recall_Disparity_Ratio': recall_disparity,
    'Boundary_Accuracy': round(acc_b, 4),
    'Interior_Accuracy': round(acc_i, 4),
    'Boundary_Disparity_Ratio': boundary_disparity,
    'Length_Disparity_Ratio': length_disparity
}])
live_fairness_df.to_csv(OUTPUT_DIR / 'live_fairness_audit.csv', index=False)
display(live_fairness_df)

## 10. Live Statistical Significance vs Baseline (McNemar & Paired Bootstrap)

In [ ]:
# Load Phase 3 baseline predictions if present, or compare against text-only
baseline_file = Path('results_phase4_live/baseline_preds.npy')
if not baseline_file.exists():
    print("[*] Generating live dummy/approximate baseline or specify path to Phase 3 predictions:")
    # Example: If Phase 3 baseline test predictions are available in Drive
    p_base = y_pred.copy()
    # Simulate or load real: np.load('/content/drive/MyDrive/F26-07_outputs/test_preds_bio_clinicalbert.npy')
else:
    p_base = np.load(str(baseline_file))
    a_ok = (p_base == y_test)
    b_ok = (y_pred == y_test)
    n_a = int(np.sum(a_ok & ~b_ok))
    n_b = int(np.sum(~a_ok & b_ok))
    p_val = float(binomtest(n_a, n_a + n_b, 0.5).pvalue)
    print(f"McNemar Test: Only Baseline Right={n_a}, Only Improved Right={n_b}, p-value={p_val}")